# Resume CelebA Training

Continue training from the saved CelebA checkpoint for 10 more epochs.

Default behavior:
- loads celeba_embedding_best.pt
- starts a fresh optimizer/scheduler from those weights
- keeps progress bars, speed, and ETA visible during training

In [1]:
from pathlib import Path
import random
import time
from collections import defaultdict

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
print("device:", device)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

device: cuda
torch: 2.6.0+cu124
cuda available: True


In [2]:
DATASET_ROOT = Path(r"C:\DSP\img_align_celeba")
IMAGE_DIR = DATASET_ROOT / "img_align_celeba"
IDENTITY_FILE = DATASET_ROOT / "Anno" / "identity_CelebA.txt"
CHECKPOINT_DIR = Path(r"C:\DSP\checkpoints")
CHECKPOINT_PATH = CHECKPOINT_DIR / "celeba_embedding_best.pt"

EXTRA_EPOCHS = 10
BATCH_SIZE = 32 if device.type == "cuda" else 8
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
TRAIN_SPLIT_RATIO = 0.9
SEED = 42
USE_PRETRAINED = False
NUM_WORKERS = 0
PIN_MEMORY = device.type == "cuda"
SHOW_EVERY = 50
RESET_OPTIMIZER = True

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

assert IMAGE_DIR.exists(), IMAGE_DIR
assert IDENTITY_FILE.exists(), IDENTITY_FILE
assert CHECKPOINT_PATH.exists(), CHECKPOINT_PATH

print("images:", IMAGE_DIR)
print("identities:", IDENTITY_FILE)
print("checkpoint:", CHECKPOINT_PATH)
print("batch size:", BATCH_SIZE)
print("num workers:", NUM_WORKERS)
print("reset optimizer:", RESET_OPTIMIZER)

images: C:\DSP\img_align_celeba\img_align_celeba
identities: C:\DSP\img_align_celeba\Anno\identity_CelebA.txt
checkpoint: C:\DSP\checkpoints\celeba_embedding_best.pt
batch size: 32
num workers: 0
reset optimizer: True


In [3]:
def load_identity_records(identity_file, image_dir):
    grouped = defaultdict(list)
    with open(identity_file, "r", encoding="utf-8") as f:
        for line in f:
            image_name, identity = line.strip().split()
            image_path = image_dir / image_name
            if image_path.exists():
                grouped[int(identity)].append(image_path)

    identity_ids = sorted(grouped)
    records = []
    reindexed = {identity_id: idx for idx, identity_id in enumerate(identity_ids)}
    for identity_id in identity_ids:
        for image_path in grouped[identity_id]:
            records.append((image_path, reindexed[identity_id], identity_id))
    return records, reindexed


def split_within_identity(records, train_ratio=0.9, seed=42):
    rng = random.Random(seed)
    grouped = defaultdict(list)
    for image_path, class_idx, original_identity in records:
        grouped[class_idx].append((image_path, class_idx, original_identity))

    train_records = []
    val_records = []
    for _, items in grouped.items():
        items = items[:]
        rng.shuffle(items)

        if len(items) == 1:
            train_records.extend(items)
            continue

        cutoff = int(len(items) * train_ratio)
        cutoff = min(max(cutoff, 1), len(items) - 1)
        train_records.extend(items[:cutoff])
        val_records.extend(items[cutoff:])

    return train_records, val_records


class CelebAIdentityDataset(Dataset):
    def __init__(self, records, transform):
        self.records = records
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        image_path, class_idx, original_identity = self.records[idx]
        image = Image.open(image_path).convert("RGB")
        image = self.transform(image)
        return image, class_idx, str(image_path), original_identity


train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

records, id_map = load_identity_records(IDENTITY_FILE, IMAGE_DIR)
train_records, val_records = split_within_identity(records, train_ratio=TRAIN_SPLIT_RATIO, seed=SEED)

train_dataset = CelebAIdentityDataset(train_records, train_transform)
val_dataset = CelebAIdentityDataset(val_records, eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

print("num classes:", len(id_map))
print("train images:", len(train_dataset))
print("val images:", len(val_dataset))

num classes: 10177
train images: 178978
val images: 23621


In [4]:
class FaceEmbeddingCNN(nn.Module):
    def __init__(self, embedding_dim, num_classes, use_pretrained=False):
        super().__init__()
        weights = models.ResNet18_Weights.DEFAULT if use_pretrained else None
        backbone = models.resnet18(weights=weights)
        in_features = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.backbone = backbone
        self.embedding = nn.Sequential(
            nn.Linear(in_features, embedding_dim),
            nn.BatchNorm1d(embedding_dim),
            nn.ReLU(inplace=True),
        )
        self.classifier = nn.Linear(embedding_dim, num_classes)

    def forward(self, x):
        features = self.backbone(x)
        embedding = self.embedding(features)
        normalized_embedding = nn.functional.normalize(embedding, p=2, dim=1)
        logits = self.classifier(embedding)
        return normalized_embedding, logits

In [5]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
embedding_dim = checkpoint.get("embedding_dim", 256)
image_size = checkpoint.get("image_size", 224)
source_epoch = int(checkpoint.get("epoch", 0))
end_epoch = source_epoch + EXTRA_EPOCHS
history = list(checkpoint.get("history", []))
best_val_loss = min((row["val_loss"] for row in history), default=float("inf"))

assert len(id_map) == checkpoint.get("num_classes", len(id_map)), "class count mismatch with checkpoint"
assert image_size == 224, f"expected 224 image size but got {image_size}"

model = FaceEmbeddingCNN(embedding_dim=embedding_dim, num_classes=len(id_map), use_pretrained=USE_PRETRAINED).to(device)
model.load_state_dict(checkpoint["model_state_dict"])

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EXTRA_EPOCHS)

if not RESET_OPTIMIZER:
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

print("loaded weights from epoch:", source_epoch)
print("continuing through epoch:", end_epoch)
print("best val loss so far:", f"{best_val_loss:.4f}")
print("learning rate:", optimizer.param_groups[0]["lr"])
model

loaded weights from epoch: 10
continuing through epoch: 20
best val loss so far: 5.6100
learning rate: 0.001


FaceEmbeddingCNN(
  (backbone): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True,

In [6]:
smoke_start = time.time()
smoke_images, smoke_labels, _, _ = next(iter(train_loader))
print("first batch shape:", tuple(smoke_images.shape), tuple(smoke_labels.shape))
print("first batch load time:", f"{time.time() - smoke_start:.2f}s")

with torch.no_grad():
    smoke_embeddings, smoke_logits = model(smoke_images[:2].to(device))
print("forward smoke test:", tuple(smoke_embeddings.shape), tuple(smoke_logits.shape))

first batch shape: (32, 3, 224, 224) (32,)
first batch load time: 0.26s
forward smoke test: (2, 256) (2, 10177)


In [7]:
def format_seconds(seconds):
    seconds = max(0, int(seconds))
    hours, rem = divmod(seconds, 3600)
    minutes, secs = divmod(rem, 60)
    if hours > 0:
        return f"{hours:d}:{minutes:02d}:{secs:02d}"
    return f"{minutes:02d}:{secs:02d}"


def run_epoch(model, loader, criterion, optimizer=None, epoch_label="?"):
    is_train = optimizer is not None
    model.train(is_train)

    total_loss = 0.0
    total_correct = 0
    total_examples = 0
    epoch_start = time.time()
    mode = "train" if is_train else "val"
    total_batches = len(loader)

    print(f"starting {epoch_label} [{mode}] | batches={total_batches}")
    progress = tqdm(
        total=total_batches,
        desc=f"{epoch_label} [{mode}]",
        leave=True,
        dynamic_ncols=True,
        mininterval=1.0,
    )

    for batch_idx, (images, labels, _, _) in enumerate(loader, start=1):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.set_grad_enabled(is_train):
            _, logits = model(images)
            loss = criterion(logits, labels)
            if is_train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_examples += batch_size

        elapsed = time.time() - epoch_start
        avg_batch_time = elapsed / batch_idx
        batches_left = total_batches - batch_idx
        eta = batches_left * avg_batch_time
        speed = total_examples / max(elapsed, 1e-6)

        if batch_idx == 1:
            print(
                f"first {mode} batch done | shape={tuple(images.shape)} | "
                f"elapsed={elapsed:.2f}s | speed={speed:.1f} img/s"
            )

        if batch_idx % SHOW_EVERY == 0 or batch_idx == total_batches:
            print(
                f"[{mode}] batch {batch_idx}/{total_batches} | "
                f"loss={total_loss / total_examples:.4f} | "
                f"acc={total_correct / total_examples:.4f} | "
                f"speed={speed:.1f} img/s | eta={format_seconds(eta)}"
            )

        progress.update(1)
        progress.set_postfix({
            "loss": f"{total_loss / total_examples:.4f}",
            "acc": f"{total_correct / total_examples:.4f}",
            "img_s": f"{speed:.1f}",
            "eta": format_seconds(eta),
        })

    progress.close()

    if total_examples == 0:
        raise RuntimeError(f"no examples were processed in {mode} epoch")

    epoch_time = time.time() - epoch_start
    return total_loss / total_examples, total_correct / total_examples, epoch_time


training_start = time.time()

for extra_epoch_idx in range(1, EXTRA_EPOCHS + 1):
    epoch = source_epoch + extra_epoch_idx
    epoch_label = f"epoch {epoch}/{end_epoch}"
    print(f"\n===== {epoch_label} =====")

    train_loss, train_acc, train_time = run_epoch(
        model,
        train_loader,
        criterion,
        optimizer=optimizer,
        epoch_label=epoch_label,
    )
    val_loss, val_acc, val_time = run_epoch(
        model,
        val_loader,
        criterion,
        optimizer=None,
        epoch_label=epoch_label,
    )
    scheduler.step()

    epoch_time = train_time + val_time
    elapsed_total = time.time() - training_start
    avg_epoch_time = elapsed_total / extra_epoch_idx
    remaining_epochs = EXTRA_EPOCHS - extra_epoch_idx
    training_eta = remaining_epochs * avg_epoch_time
    train_speed = len(train_dataset) / max(train_time, 1e-6)
    val_speed = len(val_dataset) / max(val_time, 1e-6)

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "epoch_time_sec": epoch_time,
    })

    print(
        f"epoch {epoch:02d}/{end_epoch} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} train_speed={train_speed:.1f} img/s | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_speed={val_speed:.1f} img/s | "
        f"epoch_time={format_seconds(epoch_time)} | total_eta={format_seconds(training_eta)}"
    )

    updated_checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "history": history,
        "embedding_dim": embedding_dim,
        "num_classes": len(id_map),
        "image_size": image_size,
        "id_map": id_map,
    }

    latest_path = CHECKPOINT_DIR / "celeba_embedding_latest.pt"
    torch.save(updated_checkpoint, latest_path)

    if val_loss <= best_val_loss:
        best_val_loss = val_loss
        best_path = CHECKPOINT_DIR / "celeba_embedding_best.pt"
        torch.save(updated_checkpoint, best_path)
        print("updated best checkpoint:", best_path)

print("saved to:", CHECKPOINT_DIR)


===== epoch 11/20 =====
starting epoch 11/20 [train] | batches=5594


epoch 11/20 [train]:   0%|          | 0/5594 [00:00<?, ?it/s]

first train batch done | shape=(32, 3, 224, 224) | elapsed=1.88s | speed=17.0 img/s
[train] batch 50/5594 | loss=4.7991 | acc=0.1475 | speed=98.2 img/s | eta=30:06
[train] batch 100/5594 | loss=4.9235 | acc=0.1331 | speed=105.2 img/s | eta=27:51
[train] batch 150/5594 | loss=4.9921 | acc=0.1283 | speed=107.9 img/s | eta=26:54
[train] batch 200/5594 | loss=5.0420 | acc=0.1244 | speed=110.9 img/s | eta=25:55
[train] batch 250/5594 | loss=5.0633 | acc=0.1207 | speed=113.5 img/s | eta=25:06
[train] batch 300/5594 | loss=5.0986 | acc=0.1163 | speed=115.5 img/s | eta=24:26
[train] batch 350/5594 | loss=5.1019 | acc=0.1146 | speed=117.0 img/s | eta=23:54
[train] batch 400/5594 | loss=5.1120 | acc=0.1130 | speed=118.2 img/s | eta=23:25
[train] batch 450/5594 | loss=5.1186 | acc=0.1124 | speed=119.2 img/s | eta=23:00
[train] batch 500/5594 | loss=5.1355 | acc=0.1117 | speed=119.9 img/s | eta=22:39
[train] batch 550/5594 | loss=5.1446 | acc=0.1115 | speed=120.6 img/s | eta=22:18
[train] batch 60

epoch 11/20 [val]:   0%|          | 0/739 [00:00<?, ?it/s]

first val batch done | shape=(32, 3, 224, 224) | elapsed=0.14s | speed=236.6 img/s
[val] batch 50/739 | loss=5.3781 | acc=0.1531 | speed=301.4 img/s | eta=01:13
[val] batch 100/739 | loss=5.2669 | acc=0.1653 | speed=301.2 img/s | eta=01:07
[val] batch 150/739 | loss=5.2200 | acc=0.1598 | speed=301.1 img/s | eta=01:02
[val] batch 200/739 | loss=5.2120 | acc=0.1588 | speed=302.2 img/s | eta=00:57
[val] batch 250/739 | loss=5.2183 | acc=0.1573 | speed=302.0 img/s | eta=00:51
[val] batch 300/739 | loss=5.2092 | acc=0.1581 | speed=302.1 img/s | eta=00:46
[val] batch 350/739 | loss=5.2069 | acc=0.1563 | speed=302.4 img/s | eta=00:41
[val] batch 400/739 | loss=5.2083 | acc=0.1562 | speed=302.9 img/s | eta=00:35
[val] batch 450/739 | loss=5.2125 | acc=0.1549 | speed=303.4 img/s | eta=00:30
[val] batch 500/739 | loss=5.2188 | acc=0.1552 | speed=303.9 img/s | eta=00:25
[val] batch 550/739 | loss=5.2288 | acc=0.1550 | speed=303.4 img/s | eta=00:19
[val] batch 600/739 | loss=5.2040 | acc=0.1558 | 

...

first val batch done | shape=(32, 3, 224, 224) | elapsed=0.11s | speed=280.1 img/s
[val] batch 50/739 | loss=3.4435 | acc=0.4662 | speed=332.3 img/s | eta=01:06
[val] batch 100/739 | loss=3.3547 | acc=0.4913 | speed=332.1 img/s | eta=01:01
[val] batch 150/739 | loss=3.2901 | acc=0.4979 | speed=333.5 img/s | eta=00:56
[val] batch 200/739 | loss=3.2604 | acc=0.5006 | speed=332.6 img/s | eta=00:51
[val] batch 250/739 | loss=3.2547 | acc=0.4981 | speed=333.3 img/s | eta=00:46
[val] batch 300/739 | loss=3.2557 | acc=0.4986 | speed=332.9 img/s | eta=00:42
[val] batch 350/739 | loss=3.2512 | acc=0.5019 | speed=333.2 img/s | eta=00:37
[val] batch 400/739 | loss=3.2668 | acc=0.5043 | speed=333.1 img/s | eta=00:32
[val] batch 450/739 | loss=3.2761 | acc=0.5015 | speed=333.2 img/s | eta=00:27
[val] batch 500/739 | loss=3.2778 | acc=0.5026 | speed=333.4 img/s | eta=00:22
[val] batch 550/739 | loss=3.2910 | acc=0.5003 | speed=333.4 img/s | eta=00:18
[val] batch 600/739 | loss=3.2548 | acc=0.5017 | 

epoch 20/20 [train]:   0%|          | 0/5594 [00:00<?, ?it/s]

first train batch done | shape=(32, 3, 224, 224) | elapsed=0.27s | speed=116.4 img/s
[train] batch 50/5594 | loss=0.9874 | acc=0.7831 | speed=127.8 img/s | eta=23:08
[train] batch 100/5594 | loss=1.0305 | acc=0.7684 | speed=127.3 img/s | eta=23:01
[train] batch 150/5594 | loss=1.0339 | acc=0.7652 | speed=127.5 img/s | eta=22:46
[train] batch 200/5594 | loss=1.0260 | acc=0.7672 | speed=127.6 img/s | eta=22:32
[train] batch 250/5594 | loss=1.0386 | acc=0.7650 | speed=127.6 img/s | eta=22:20
[train] batch 300/5594 | loss=1.0472 | acc=0.7628 | speed=127.6 img/s | eta=22:07
[train] batch 350/5594 | loss=1.0436 | acc=0.7643 | speed=127.6 img/s | eta=21:54
[train] batch 400/5594 | loss=1.0601 | acc=0.7595 | speed=127.8 img/s | eta=21:40
[train] batch 450/5594 | loss=1.0592 | acc=0.7603 | speed=127.7 img/s | eta=21:28
[train] batch 500/5594 | loss=1.0589 | acc=0.7602 | speed=127.7 img/s | eta=21:16
[train] batch 550/5594 | loss=1.0625 | acc=0.7600 | speed=127.6 img/s | eta=21:05
[train] batch 

epoch 20/20 [val]:   0%|          | 0/739 [00:00<?, ?it/s]

first val batch done | shape=(32, 3, 224, 224) | elapsed=0.11s | speed=294.6 img/s
[val] batch 50/739 | loss=3.4449 | acc=0.4775 | speed=342.3 img/s | eta=01:04
[val] batch 100/739 | loss=3.3449 | acc=0.4966 | speed=341.3 img/s | eta=00:59
[val] batch 150/739 | loss=3.2811 | acc=0.5035 | speed=342.1 img/s | eta=00:55
[val] batch 200/739 | loss=3.2513 | acc=0.5064 | speed=342.4 img/s | eta=00:50
[val] batch 250/739 | loss=3.2484 | acc=0.5039 | speed=342.9 img/s | eta=00:45
[val] batch 300/739 | loss=3.2417 | acc=0.5043 | speed=343.2 img/s | eta=00:40
[val] batch 350/739 | loss=3.2371 | acc=0.5077 | speed=343.3 img/s | eta=00:36
[val] batch 400/739 | loss=3.2503 | acc=0.5104 | speed=343.4 img/s | eta=00:31
[val] batch 450/739 | loss=3.2558 | acc=0.5086 | speed=344.0 img/s | eta=00:26
[val] batch 500/739 | loss=3.2553 | acc=0.5092 | speed=344.1 img/s | eta=00:22
[val] batch 550/739 | loss=3.2688 | acc=0.5067 | speed=344.1 img/s | eta=00:17
[val] batch 600/739 | loss=3.2330 | acc=0.5082 | 